# Project 7 — Trust After AI Failure
## Do actionable error disclosures improve appropriate reliance after AI failure?

**Target journal:** *Computers in Human Behavior* (Elsevier; subscription/hybrid)  
**Research focus:** Human–computer interaction, trust calibration, appropriate reliance, and recovery after AI errors.

### Project proposal and research questions
This notebook operationalizes the supplied project proposal. The proposed experiment compares three disclosure conditions after AI advice errors:

1. **Generic apology** — a brief acknowledgement of the error.
2. **Calibrated uncertainty disclosure** — acknowledgement plus an explanation that the advice may be unreliable.
3. **Actionable correction-and-review route** — acknowledgement, corrected information, and an opportunity to review the evidence.

Participants encounter both correct and incorrect AI advice in a repeated-measures decision task. The task is hypothetical and non-consequential.

**RQ1.** Which disclosure produces the best discrimination between correct and incorrect AI advice after an error?  
**RQ2.** Do changes in reported trust track changes in appropriate reliance, or do the two diverge?  
**RQ3.** Does access to a correction-and-review route improve recovery after repeated errors?  
**RQ4.** How do initial expectations and prior AI experience moderate these effects?

### Important research and ethics boundary
The supplied proposal requires a new participant dataset. This notebook creates **synthetic demonstration data only** so that the pipeline can be tested before recruitment. Synthetic results are not evidence and must not be reported as participant findings. Recruit participants only after institutional ethics approval, informed consent, and an appropriate power analysis. Avoid recruiting classmates in a way that creates pressure or coercion.

### Execution plan
Run the notebook from top to bottom in Kaggle. For the demonstration, it creates synthetic data. To analyse real data, set `USE_REAL_DATA = True` and upload a CSV that follows the documented schema. All outputs are collected in `/kaggle/working/trust_after_ai_failure_outputs/` and zipped at the end.

## 1. Environment and reproducibility

**Purpose:** Import the analysis libraries, create output folders, record the runtime configuration, and set a random seed. The notebook is compatible with Kaggle CPU or GPU sessions.

**Research relevance:** Reproducible analysis is essential for transparent behavioral research. GPU availability is recorded, but the core statistical models are CPU-based; this is a behavioral experiment rather than a deep-learning workload.

In [ ]:
import os, sys, json, platform, warnings, zipfile, subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
SEED = 20261102
rng = np.random.default_rng(SEED)

# Kaggle path, with a local fallback for testing outside Kaggle
BASE_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUTPUT_DIR = BASE_DIR / "trust_after_ai_failure_outputs"
FIG_DIR = OUTPUT_DIR / "figures_pdf"
TABLE_DIR = OUTPUT_DIR / "tables_csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
    GPU_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else "No CUDA GPU detected"
except Exception:
    GPU_AVAILABLE, GPU_NAME = False, "PyTorch unavailable"

runtime = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gpu_available": GPU_AVAILABLE,
    "gpu_name": GPU_NAME,
    "random_seed": SEED,
    "created_utc": datetime.now(timezone.utc).isoformat()
}
print(json.dumps(runtime, indent=2))
print("GPU status:", GPU_NAME)
print("Output directory:", OUTPUT_DIR)

## 2. Study design and analysis decisions

**Purpose:** Define the experimental factors, variables, and operational definitions before inspecting results.

**Design:** Participants are assigned to one of three between-participant disclosure conditions. Each participant completes repeated trials containing correct and incorrect advice. The trial order should be randomized in the actual task, and stimulus/item identity should be retained.

**Primary behavioral outcome:** Whether the participant accepts the AI advice (`accepted = 1`) or overrides it (`accepted = 0`).

**Appropriate reliance:** Accepting correct advice and rejecting incorrect advice.  
**Overreliance:** Accepting incorrect advice.  
**Underreliance:** Rejecting correct advice.  
**Discrimination:** Difference in acceptance probability between correct and incorrect advice. Larger positive differences indicate better discrimination.

The synthetic generator below is only a software test fixture. Replace it with ethically collected participant data before substantive inference.

In [ ]:
CONDITIONS = [
    "generic_apology",
    "uncertainty_disclosure",
    "correction_review"
]
CONDITION_LABELS = {
    "generic_apology": "Generic apology",
    "uncertainty_disclosure": "Uncertainty disclosure",
    "correction_review": "Correction + review"
}
USE_REAL_DATA = False
REAL_DATA_PATH = Path("/kaggle/input")  # Change to the uploaded CSV path if needed

# Planned trial structure for the demonstration only
N_SYNTH_PARTICIPANTS = 180
N_TRIALS_PER_PARTICIPANT = 24

## 3. A priori sample-size planning — preliminary approximation

**Purpose:** Provide a transparent planning calculation for a two-sided comparison of two independent groups on a standardized outcome.

**Research relevance:** The proposal requires power-based recruitment rather than defaulting to the available classmates. This calculation is a rough planning aid, not a substitute for a design-specific simulation-based power analysis for the final mixed-effects model. The final sample size should be justified using the primary estimand, expected effect, repeated-trial structure, participant/item variability, attrition, and multiplicity plan.

In [ ]:
from statsmodels.stats.power import TTestIndPower

power_analysis = TTestIndPower()
planning_rows = []
for d in [0.30, 0.40, 0.50]:
    n_per_group = power_analysis.solve_power(
        effect_size=d, alpha=0.05, power=0.80,
        ratio=1.0, alternative="two-sided"
    )
    planning_rows.append({
        "standardized_effect_d": d,
        "power": 0.80,
        "alpha": 0.05,
        "approx_n_per_group": int(np.ceil(n_per_group)),
        "approx_total_for_three_groups": 3 * int(np.ceil(n_per_group))
    })
power_table = pd.DataFrame(planning_rows)
display(power_table)
power_table.to_csv(TABLE_DIR / "preliminary_power_planning.csv", index=False)

## 4. Synthetic demonstration dataset

**Purpose:** Create a trial-level dataset to verify the workflow and demonstrate the planned variables.

**Research relevance:** The simulated structure includes advice correctness, disclosure condition, response time, repeated trust ratings, initial expectations, prior AI experience, and error sequence. The data-generating parameters are illustrative and deliberately do not represent empirical estimates.

For the real study, replace this section's output with the anonymized dataset collected through the approved task. Do not use synthetic observations as research evidence.

In [ ]:
def make_synthetic_data(n_participants=180, n_trials=24, seed=SEED):
    local_rng = np.random.default_rng(seed)
    rows = []
    conditions = np.repeat(CONDITIONS, int(np.ceil(n_participants / 3)))[:n_participants]
    local_rng.shuffle(conditions)

    participant_intercepts = local_rng.normal(0, 0.8, n_participants)
    for p in range(n_participants):
        condition = conditions[p]
        initial_expectation = np.clip(local_rng.normal(3.5, 0.9), 1, 5)
        prior_ai_experience = np.clip(local_rng.normal(3.0, 1.2), 1, 5)
        trust = np.clip(initial_expectation, 1, 5)
        correctness_sequence = np.array([1, 0] * (n_trials // 2))
        if n_trials % 2:
            correctness_sequence = np.append(correctness_sequence, 1)
        local_rng.shuffle(correctness_sequence)

        consecutive_errors = 0
        for t, correct in enumerate(correctness_sequence, start=1):
            if correct == 0:
                consecutive_errors += 1
            else:
                consecutive_errors = 0

            # Illustrative acceptance probabilities; not empirical estimates.
            condition_effect = {
                "generic_apology": 0.0,
                "uncertainty_disclosure": -0.25,
                "correction_review": -0.45
            }[condition]
            logit = (
                0.25 + 0.95 * (correct - 0.5)
                + condition_effect * (1 - correct)
                + 0.10 * (initial_expectation - 3)
                + 0.08 * (prior_ai_experience - 3)
                + participant_intercepts[p]
            )
            prob_accept = 1 / (1 + np.exp(-logit))
            accepted = int(local_rng.binomial(1, prob_accept))
            response_time = max(0.5, local_rng.lognormal(1.25 + 0.12 * (1-correct), 0.35))

            # Trust ratings are repeated self-reports (1–5).
            trust_change = (
                0.08 if correct else -0.20
            )
            if condition == "correction_review" and not correct:
                trust_change += 0.10
            elif condition == "uncertainty_disclosure" and not correct:
                trust_change += 0.03
            trust = float(np.clip(trust + trust_change + local_rng.normal(0, 0.25), 1, 5))
            rows.append({
                "participant_id": f"P{p+1:04d}",
                "trial_id": t,
                "item_id": f"I{(t-1)%12+1:02d}",
                "condition": condition,
                "advice_correct": int(correct),
                "accepted": accepted,
                "response_time_seconds": response_time,
                "trust_rating": trust,
                "initial_expectation": initial_expectation,
                "prior_ai_experience": prior_ai_experience,
                "error_sequence": consecutive_errors,
                "data_source": "SYNTHETIC_DEMONSTRATION"
            })
    return pd.DataFrame(rows)

if USE_REAL_DATA:
    # Set REAL_DATA_PATH to the exact CSV file uploaded to Kaggle.
    if REAL_DATA_PATH.is_dir():
        csv_candidates = list(REAL_DATA_PATH.rglob("*.csv"))
        if len(csv_candidates) != 1:
            raise ValueError(f"Expected exactly one CSV in {REAL_DATA_PATH}; found {len(csv_candidates)}. Set REAL_DATA_PATH to the desired file.")
        DATA_PATH = csv_candidates[0]
    else:
        DATA_PATH = REAL_DATA_PATH
    df = pd.read_csv(DATA_PATH)
else:
    df = make_synthetic_data(N_SYNTH_PARTICIPANTS, N_TRIALS_PER_PARTICIPANT)

print("Dataset shape:", df.shape)
display(df.head())

## 5. Data dictionary, validation, and anonymization checks

**Purpose:** Check required columns, permissible values, missingness, and participant-level consistency.

**Research relevance:** Trial-level data must preserve the nesting of trials within participants and items. Participant IDs should be pseudonymous; direct identifiers, contact details, IP addresses, and unnecessary sensitive attributes should not be included in the analysis file.

In [ ]:
REQUIRED_COLUMNS = [
    "participant_id", "trial_id", "item_id", "condition",
    "advice_correct", "accepted", "response_time_seconds",
    "trust_rating", "initial_expectation", "prior_ai_experience"
]
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df.copy()
df["condition"] = pd.Categorical(df["condition"], categories=CONDITIONS)
for col in ["advice_correct", "accepted"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    if not df[col].dropna().isin([0, 1]).all():
        raise ValueError(f"{col} must contain only 0/1 values.")
for col in ["trust_rating", "initial_expectation", "prior_ai_experience", "response_time_seconds"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Missing values:")
display(df[REQUIRED_COLUMNS].isna().sum().rename("missing_count").to_frame())
print("Participants:", df["participant_id"].nunique())
print("Items:", df["item_id"].nunique())
print("Conditions:", df["condition"].value_counts(dropna=False))
print("Data source labels:", df["data_source"].value_counts() if "data_source" in df else "Not supplied")

## 6. Derived behavioral outcomes

**Purpose:** Derive appropriate reliance, overreliance, and underreliance without conflating them.

**Research relevance:** A single overall acceptance rate is insufficient. High acceptance can be beneficial when advice is correct but harmful when advice is incorrect. These separate outcomes directly support RQ1 and the project's central distinction between trust and appropriate reliance.

In [ ]:
df["appropriate_reliance"] = (
    ((df["advice_correct"] == 1) & (df["accepted"] == 1)) |
    ((df["advice_correct"] == 0) & (df["accepted"] == 0))
).astype(int)
df["overreliance"] = ((df["advice_correct"] == 0) & (df["accepted"] == 1)).astype(int)
df["underreliance"] = ((df["advice_correct"] == 1) & (df["accepted"] == 0)).astype(int)

participant_summary = df.groupby(["participant_id", "condition"], observed=True).agg(
    acceptance_rate=("accepted", "mean"),
    trust_mean=("trust_rating", "mean"),
    appropriate_reliance=("appropriate_reliance", "mean"),
    overreliance=("overreliance", "mean"),
    underreliance=("underreliance", "mean"),
    initial_expectation=("initial_expectation", "first"),
    prior_ai_experience=("prior_ai_experience", "first")
).reset_index()

display(participant_summary.head())
participant_summary.to_csv(TABLE_DIR / "participant_level_summary.csv", index=False)

## 7. Descriptive statistics and trial-level overview

**Purpose:** Summarize participant counts, trial counts, acceptance, trust, and response time by condition and advice correctness.

**Research relevance:** These descriptive results establish the observed pattern before modeling. They are not causal conclusions on their own. Inspect cell counts and missingness before interpreting condition differences.

In [ ]:
descriptive = df.groupby(["condition", "advice_correct"], observed=True).agg(
    participants=("participant_id", "nunique"),
    trials=("accepted", "size"),
    acceptance_rate=("accepted", "mean"),
    trust_mean=("trust_rating", "mean"),
    trust_sd=("trust_rating", "std"),
    median_response_time=("response_time_seconds", "median"),
    appropriate_reliance=("appropriate_reliance", "mean"),
    overreliance=("overreliance", "mean"),
    underreliance=("underreliance", "mean")
).reset_index()
display(descriptive)
descriptive.to_csv(TABLE_DIR / "condition_correctness_descriptives.csv", index=False)

## 8. RQ1 — Discrimination between correct and incorrect advice

**Purpose:** Estimate acceptance rates separately for correct and incorrect advice and calculate the discrimination gap (acceptance of correct advice minus acceptance of incorrect advice).

**Research relevance:** A larger gap indicates more selective reliance. Also report overreliance and underreliance separately: a condition may reduce overreliance while inadvertently increasing underreliance.

The logistic regression includes condition, advice correctness, and their interaction, with participant-clustered robust standard errors. Item fixed effects are included where supported by the data. This is a pragmatic model; a preregistered final analysis should consider a generalized linear mixed-effects model with participant and item random effects.

In [ ]:
formula = "accepted ~ C(condition) * advice_correct + initial_expectation + prior_ai_experience + C(item_id)"
rq1_model = smf.glm(formula, data=df, family=sm.families.Binomial()).fit(
    cov_type="cluster", cov_kwds={"groups": df["participant_id"]}
)
print(rq1_model.summary())

rq1_table = pd.DataFrame({
    "term": rq1_model.params.index,
    "log_odds_estimate": rq1_model.params.values,
    "odds_ratio": np.exp(rq1_model.params.values),
    "p_value": rq1_model.pvalues.values
})
display(rq1_table)
rq1_table.to_csv(TABLE_DIR / "rq1_logistic_model.csv", index=False)

rq1_rates = df.groupby(["condition", "advice_correct"], observed=True)["accepted"].mean().unstack()
rq1_rates["discrimination_gap"] = rq1_rates.get(1) - rq1_rates.get(0)
rq1_rates = rq1_rates.rename(columns={0: "acceptance_incorrect_advice", 1: "acceptance_correct_advice"}).reset_index()
display(rq1_rates)
rq1_rates.to_csv(TABLE_DIR / "rq1_discrimination_by_condition.csv", index=False)

## 9. RQ2 — Do reported trust and appropriate reliance move together?

**Purpose:** Compare condition-level trust ratings with behavioral reliance outcomes and estimate the association between trust and appropriate reliance.

**Research relevance:** Self-reported trust is not interchangeable with observed decision quality. The correlation is descriptive and does not establish that trust causes reliance or vice versa. A participant-clustered model is used for the trial-level association.

In [ ]:
rq2_corr = participant_summary[["trust_mean", "appropriate_reliance", "overreliance", "underreliance"]].corr()
display(rq2_corr)
rq2_corr.to_csv(TABLE_DIR / "rq2_participant_level_correlations.csv")

rq2_model = smf.ols(
    "appropriate_reliance ~ trust_rating + C(condition) + advice_correct",
    data=df
).fit(cov_type="cluster", cov_kwds={"groups": df["participant_id"]})
print(rq2_model.summary())
pd.DataFrame({
    "term": rq2_model.params.index,
    "estimate": rq2_model.params.values,
    "p_value": rq2_model.pvalues.values
}).to_csv(TABLE_DIR / "rq2_trust_behavior_model.csv", index=False)

## 10. RQ3 — Recovery after repeated errors

**Purpose:** Examine behavior and trust after the first, second, and subsequent consecutive incorrect-advice events.

**Research relevance:** The correction-and-review route is expected to be most relevant to recovery after repeated errors. The analysis compares the trend by condition while retaining the sequence measure. Interpret cautiously if there are few observations at higher error-sequence counts.

In [ ]:
error_trials = df[df["error_sequence"] > 0].copy()
if not error_trials.empty:
    recovery = error_trials.groupby(["condition", "error_sequence"], observed=True).agg(
        n_trials=("accepted", "size"),
        acceptance_rate=("accepted", "mean"),
        trust_mean=("trust_rating", "mean"),
        overreliance=("overreliance", "mean")
    ).reset_index()
    display(recovery)
    recovery.to_csv(TABLE_DIR / "rq3_recovery_after_consecutive_errors.csv", index=False)

    # Restrict plotted sequence to levels with a minimally useful number of trials.
    plot_recovery = recovery[recovery["n_trials"] >= 10]
    if not plot_recovery.empty:
        plt.figure(figsize=(9, 5))
        sns.lineplot(data=plot_recovery, x="error_sequence", y="overreliance",
                     hue="condition", marker="o")
        plt.title("Overreliance across consecutive incorrect-advice events")
        plt.xlabel("Consecutive incorrect-advice event")
        plt.ylabel("Overreliance rate")
        plt.ylim(0, 1)
        plt.tight_layout()
        plt.savefig(FIG_DIR / "rq3_recovery_overreliance.pdf", bbox_inches="tight")
        plt.show()
else:
    print("No incorrect-advice sequences available for RQ3.")

## 11. RQ4 — Initial expectations and prior AI experience

**Purpose:** Test whether baseline expectations and prior AI experience are associated with reliance, and whether these relationships differ across disclosure conditions.

**Research relevance:** These moderators were specified in the proposal. The model includes interactions with condition. Because the measures are observational baseline characteristics, moderation should not be interpreted as causal effects of experience or expectations.

In [ ]:
rq4_formula = (
    "accepted ~ C(condition) * advice_correct "
    "+ C(condition):initial_expectation "
    "+ C(condition):prior_ai_experience "
    "+ initial_expectation + prior_ai_experience + C(item_id)"
)
rq4_model = smf.glm(rq4_formula, data=df, family=sm.families.Binomial()).fit(
    cov_type="cluster", cov_kwds={"groups": df["participant_id"]}
)
print(rq4_model.summary())
rq4_table = pd.DataFrame({
    "term": rq4_model.params.index,
    "log_odds_estimate": rq4_model.params.values,
    "odds_ratio": np.exp(rq4_model.params.values),
    "p_value": rq4_model.pvalues.values
})
rq4_table.to_csv(TABLE_DIR / "rq4_moderation_model.csv", index=False)
display(rq4_table)

## 12. Publication-quality figures

**Purpose:** Generate readable, clearly labelled figures with consistent styling and export each figure as a PDF.

**Research relevance:** The figures focus on behavioral discrimination, trust versus appropriate reliance, and over/underreliance. All plots are based on the currently loaded dataset; if it is synthetic, the figure titles and interpretation must remain explicitly marked as demonstration-only.

In [ ]:
sns.set_theme(style="whitegrid", context="talk")
source_label = "SYNTHETIC DEMONSTRATION — NOT PARTICIPANT EVIDENCE" if (
    "data_source" in df and df["data_source"].eq("SYNTHETIC_DEMONSTRATION").all()
) else "Participant dataset"

# Figure 1: Acceptance by advice correctness and condition
plt.figure(figsize=(10, 6))
sns.barplot(data=df, x="condition", y="accepted", hue="advice_correct",
            errorbar=("ci", 95), palette="Set2")
plt.xticks(range(len(CONDITIONS)), [CONDITION_LABELS[c] for c in CONDITIONS], rotation=12, ha="right")
plt.xlabel("Disclosure condition")
plt.ylabel("Probability of accepting AI advice")
plt.title("Reliance discrimination by disclosure condition")
plt.legend(title="Advice correct", labels=["Incorrect", "Correct"])
plt.suptitle(source_label, fontsize=10, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "figure1_reliance_discrimination.pdf", bbox_inches="tight")
plt.show()

# Figure 2: Trust and appropriate reliance by condition (different scales shown separately)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=participant_summary, x="condition", y="trust_mean", ax=axes[0],
            errorbar=("ci", 95), color="#6BAED6")
sns.barplot(data=participant_summary, x="condition", y="appropriate_reliance", ax=axes[1],
            errorbar=("ci", 95), color="#74C476")
for ax in axes:
    ax.set_xticks(range(len(CONDITIONS)))
    ax.set_xticklabels([CONDITION_LABELS[c] for c in CONDITIONS], rotation=15, ha="right")
    ax.set_xlabel("Disclosure condition")
axes[0].set_ylabel("Mean reported trust (1–5)")
axes[0].set_title("Self-reported trust")
axes[1].set_ylabel("Appropriate reliance rate")
axes[1].set_ylim(0, 1)
axes[1].set_title("Observed appropriate reliance")
fig.suptitle("Trust and behavior are distinct outcomes\n" + source_label, fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "figure2_trust_vs_behavior.pdf", bbox_inches="tight")
plt.show()

# Figure 3: Overreliance and underreliance
behavior_long = participant_summary.melt(
    id_vars=["participant_id", "condition"],
    value_vars=["overreliance", "underreliance"],
    var_name="error_type", value_name="rate"
)
plt.figure(figsize=(10, 6))
sns.barplot(data=behavior_long, x="condition", y="rate", hue="error_type",
            errorbar=("ci", 95), palette="Set1")
plt.xticks(range(len(CONDITIONS)), [CONDITION_LABELS[c] for c in CONDITIONS], rotation=12, ha="right")
plt.xlabel("Disclosure condition")
plt.ylabel("Participant-level rate")
plt.title("Overreliance and underreliance must be assessed separately")
plt.suptitle(source_label, fontsize=10, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "figure3_over_under_reliance.pdf", bbox_inches="tight")
plt.show()

## 13. Export final analysis tables

**Purpose:** Save the main descriptive and inferential outputs as CSV files in one location.

**Research relevance:** Machine-readable tables support reproducibility, review, and later manuscript preparation. The exported tables should be regenerated from the final cleaned participant dataset after the preregistered exclusions and analysis decisions have been applied.

In [ ]:
# Export core datasets and summaries for auditability.
df.to_csv(TABLE_DIR / "analysis_dataset_trial_level.csv", index=False)
participant_summary.to_csv(TABLE_DIR / "participant_level_summary.csv", index=False)
descriptive.to_csv(TABLE_DIR / "condition_correctness_descriptives.csv", index=False)

# A compact analysis manifest records the data provenance and key settings.
manifest = {
    "project": "Project 7 — Trust After AI Failure",
    "target_journal": "Computers in Human Behavior",
    "data_source": (
        "SYNTHETIC DEMONSTRATION — NOT PARTICIPANT EVIDENCE"
        if "data_source" in df and df["data_source"].eq("SYNTHETIC_DEMONSTRATION").all()
        else "User-supplied participant dataset; provenance must be documented separately"
    ),
    "n_participants": int(df["participant_id"].nunique()),
    "n_trials": int(len(df)),
    "conditions": CONDITIONS,
    "random_seed": SEED,
    "gpu_available": GPU_AVAILABLE,
    "analysis_note": "Core statistical models use CPU; GPU is not required."
}
with open(OUTPUT_DIR / "analysis_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))

## 14. Package all outputs into one ZIP file

**Purpose:** Collect every PDF figure, CSV table, and analysis manifest into one compressed archive.

**Research relevance:** A single archive makes it easier to retain and share the reproducible analysis outputs. Do not include identifiable participant information. For an actual study, follow institutional data-management and access rules before sharing any participant-level file.

In [ ]:
zip_path = BASE_DIR / "trust_after_ai_failure_results.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_DIR.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(OUTPUT_DIR.parent))

print("ZIP created:", zip_path)
print("ZIP size (MB):", round(zip_path.stat().st_size / (1024**2), 3))
print("\nIncluded files:")
with zipfile.ZipFile(zip_path, "r") as zf:
    for name in zf.namelist():
        print(" -", name)

## 15. Interpretation and reporting checklist

Before writing the paper or drawing conclusions, confirm:

- [ ] Ethics approval and informed consent were obtained before recruitment.
- [ ] The task, randomization, exclusions, primary outcome, and analysis plan were preregistered.
- [ ] Sample size was justified for the actual primary model and expected effect.
- [ ] The participant and item structure is retained in the dataset.
- [ ] Overreliance and underreliance are reported separately.
- [ ] Trust ratings are not treated as a substitute for behavioral reliance.
- [ ] Missing data, exclusions, and deviations are transparently reported.
- [ ] All synthetic data and synthetic outputs have been removed from empirical reporting.
- [ ] The manuscript's journal metrics and recent-paper details are independently verified at submission time.

**GPU note:** Enabling a Kaggle GPU is fine, but it will not materially speed up the GLM/OLS analyses in this notebook. The main computational work is small-scale statistical modeling. Do not add a neural network merely to justify GPU usage; it would not answer the stated research questions.